# Trials criterion pilot: reproducible audit
This companion reads the pinned raw source and saved development predictions. It does not run model inference, change reference labels, or score held-out patients. Run the preparation and evaluation commands in `docs/TRIALS_CRITERION_PILOT.md` first.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'config/trials_criterion_pilot.json').exists()
prepared = ROOT / 'preprocess/trials_criterion_pilot'
results = ROOT / 'results/trials_criterion_pilot'
raw = pd.read_parquet(ROOT / 'dataset/trialgpt_criterion/raw/data/train-00000-of-00001.parquet')
print({'source_rows': len(raw), 'patients': raw.patient_id.nunique(), 'trials': raw.trial_id.nunique()})
print(raw.isna().sum().to_string())
assert raw.annotation_id.is_unique
assert not raw.duplicated(['patient_id', 'trial_id', 'criterion_type', 'criterion_text']).any()


## Label and evidence audit
The row-level source training flag is not a patient-level split. Empty expert evidence is an annotation-policy review flag, not permission to relabel the case as unknown. The criterion with missing text is excluded from modeling and preserved in quarantine.

In [ ]:
overlap = int((raw.groupby('patient_id').training.nunique() > 1).sum())
print('Patients crossing the source training flag:', overlap)
print(raw.groupby(['criterion_type', 'expert_eligibility']).size().to_string())
print('Source rows without annotated evidence:', raw.expert_sentences.eq('[]').sum())
audit = json.loads((prepared / 'data_quality_audit.json').read_text())
print('Usable-data review flags:', audit['flag_counts'])
print('Quarantine:', (prepared / 'quarantine.jsonl').read_text())
assert overlap == audit['patients_crossing_source_training_flag']
assert len(raw) - audit['quarantined_rows'] == audit['rows']


## Patient-disjoint partitions
This reads only the split manifest. Held-out labels and model predictions are not opened in this notebook. One trial identifier is shared between training and holdout; the design is not fully protocol-disjoint.

In [ ]:
split = json.loads((prepared / 'split_manifest.json').read_text())
groups = {name: set(ids) for name, ids in split['patients'].items()}
assert not groups['train'] & groups['development']
assert not groups['train'] & groups['holdout']
assert not groups['development'] & groups['holdout']
print(pd.DataFrame(split['summary']).T[['patients', 'rows', 'trials']].to_string())
print('Trial overlap:', split['trial_overlap'])
review = pd.read_csv(prepared / 'review/development_review_blinded.csv')
assert set(review.patient_id) <= groups['development']
print('Review cases:', len(review), 'review status counts:', review.review_status.value_counts().to_dict())


## Recompute saved development metrics
All four classes remain in the macro-F1 denominator. Each prediction is joined one-to-one with a development reference. Historical GPT-4 outputs are not counted as fresh local runs.

In [ ]:
pred = pd.read_json(results / 'development_predictions.jsonl', lines=True)
refs = pd.read_json(prepared / 'development/references.jsonl', lines=True)
assert refs.record_id.is_unique
assert not pred.duplicated(['system', 'record_id']).any()
joined = pred.merge(refs[['record_id', 'constraint_status']], on='record_id', how='left', validate='many_to_one')
assert joined.constraint_status.notna().all()
labels = ['met', 'not_met', 'unknown', 'not_applicable']
recomputed = []
for system, rows in joined.groupby('system'):
    recomputed.append({'system': system, 'n': len(rows), 'accuracy': accuracy_score(rows.constraint_status, rows.predicted_status), 'macro_f1': f1_score(rows.constraint_status, rows.predicted_status, labels=labels, average='macro', zero_division=0)})
print(pd.DataFrame(recomputed).to_string(index=False))
saved = json.loads((results / 'development_results.json').read_text())
assert saved['holdout_scored'] is False
for row in recomputed:
    assert abs(row['macro_f1'] - saved['metrics'][row['system']]['macro_f1']) < 1e-12
print('Independent verification:', json.loads((results / 'verification.json').read_text()))


## Interpretation and next decision
The first local classifier has a large train-to-development gap. Identical classifications across the ontology variants do not establish a general null ontology effect. Use a stronger criterion evaluator or reduce feature complexity through training-only patient-group cross-validation; adjudicate the blinded review set before defining a new evidence-policy target. Do not tune on the prepared holdout.